# Single-Class YOLO Training - Micro Dataset (Kaggle)

This notebook trains YOLO for one class: `microplastic`, using your augmented dataset
from `data/micro/yolo_single_aug` uploaded to Kaggle.

It follows the same logic as your reference notebook:
- auto-detect dataset under `/kaggle/input`
- copy to `/kaggle/working`
- fix `dataset.yaml` path
- train + validate + export

## 1. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.1.0', 'pyyaml'], check=True)

import ultralytics
print('Ultralytics:', ultralytics.__version__)

## 2. GPU Check + Reproducibility

In [ ]:
import os
import random
import numpy as np
import torch
import psutil

assert torch.cuda.is_available(), 'No GPU detected. In Kaggle set Accelerator to GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('RAM GB:', f"{psutil.virtual_memory().total / 1e9:.1f}")
print('Disk free GB:', f"{psutil.disk_usage('/kaggle/working').free / 1e9:.1f}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
print('Seed:', SEED)

## 3. Auto-Discover Dataset and Copy to Working Directory

Expected dataset structure (uploaded to Kaggle):
- dataset.yaml
- images/train, images/val
- labels/train, labels/val

Tip: name your Kaggle dataset with `micro` and/or `single` in its path for best auto-detection.

In [ ]:
import os
import shutil
import yaml
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
LOCAL_DATASET_PATH = Path('/kaggle/working/dataset_micro_single_aug')
OUTPUT_PATH = Path('/kaggle/working/experiments/yolo')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print('Input folders:', sorted([p.name for p in INPUT_ROOT.iterdir()]))

candidates = []
for yml in INPUT_ROOT.rglob('dataset.yaml'):
    parent = yml.parent
    if (parent / 'images').exists() and (parent / 'labels').exists():
        score = 0
        pstr = str(parent).lower()
        if 'micro' in pstr:
            score += 2
        if 'single' in pstr:
            score += 2
        if 'aug' in pstr:
            score += 1
        candidates.append((score, parent))

assert candidates, 'No valid YOLO dataset found. Need dataset.yaml + images/ + labels/ under /kaggle/input.'
candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
src = candidates[0][1]
print('Using source:', src)

if LOCAL_DATASET_PATH.exists():
    # self-heal stale copies
    if not (LOCAL_DATASET_PATH / 'dataset.yaml').exists() or not (LOCAL_DATASET_PATH / 'images' / 'train').exists():
        shutil.rmtree(LOCAL_DATASET_PATH)

if not LOCAL_DATASET_PATH.exists():
    shutil.copytree(str(src), str(LOCAL_DATASET_PATH))
    print('Copied to:', LOCAL_DATASET_PATH)
else:
    print('Already exists and valid:', LOCAL_DATASET_PATH)

YAML_PATH = LOCAL_DATASET_PATH / 'dataset.yaml'
assert YAML_PATH.exists(), f'dataset.yaml missing at {YAML_PATH}'

with open(YAML_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(LOCAL_DATASET_PATH)
with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('Updated dataset.yaml path ->', cfg['path'])
print('Class config:', cfg.get('names'))

## 4. Train YOLO (Single-Class microplastic)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import datetime
import shutil

MODEL = 'yolov8m.pt'
IMGSZ = 1280
BATCH_SIZE = 2
EPOCHS = 200
PATIENCE = 50
EXPERIMENT = 'mp_yolov8m_single_class_micro'
BACKUP_EVERY = 50
BACKUP_ROOT = Path('/kaggle/working/backups/yolo_micro')

def auto_backup(trainer):
    epoch = trainer.epoch + 1
    if epoch % BACKUP_EVERY != 0:
        return
    weights_dir = Path(trainer.save_dir) / 'weights'
    backup_dir = BACKUP_ROOT / f'epoch_{epoch:04d}'
    backup_dir.mkdir(parents=True, exist_ok=True)
    for name in ('best.pt', 'last.pt'):
        src = weights_dir / name
        if src.exists():
            shutil.copy2(src, backup_dir / name)
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[AUTO-BACKUP] epoch {epoch} -> {backup_dir} ({ts})')

model = YOLO(MODEL)
model.add_callback('on_train_epoch_end', auto_backup)

print('=' * 70)
print('Training YOLO micro single-class')
print('data:', str(YAML_PATH))
print('project:', str(OUTPUT_PATH))
print('name:', EXPERIMENT)
print('=' * 70)

results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=0,
    workers=2,
    seed=SEED,
    deterministic=True,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.01,
    weight_decay=5e-4,
    warmup_epochs=5.0,
    cos_lr=True,
    patience=PATIENCE,
    augment=True,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    close_mosaic=20,
    fliplr=0.5,
    flipud=0.5,
    degrees=15.0,
    translate=0.2,
    scale=0.5,
    shear=5.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    amp=True,
    cache='disk',
    multi_scale=False,
    project=str(OUTPUT_PATH),
    name=EXPERIMENT,
    exist_ok=True,
    save=True,
    save_period=25,
    plots=True
)

print('Best:', OUTPUT_PATH / EXPERIMENT / 'weights' / 'best.pt')
print('Last:', OUTPUT_PATH / EXPERIMENT / 'weights' / 'last.pt')

## 5. Validate Best Model

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST = OUTPUT_PATH / EXPERIMENT / 'weights' / 'best.pt'
if not BEST.exists():
    backups = sorted(BACKUP_ROOT.glob('epoch_*/best.pt')) if BACKUP_ROOT.exists() else []
    assert backups, 'best.pt not found in main or backup paths.'
    BEST = backups[-1]

print('Validating:', BEST)
m = YOLO(str(BEST))
metrics = m.val(data=str(YAML_PATH), imgsz=IMGSZ, batch=BATCH_SIZE, conf=0.001, iou=0.6, plots=True)

print('mAP50:', f"{metrics.box.map50:.4f}")
print('mAP50-95:', f"{metrics.box.map:.4f}")
print('Precision:', f"{metrics.box.mp:.4f}")
print('Recall:', f"{metrics.box.mr:.4f}")

## 6. Export and Download

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

BEST = OUTPUT_PATH / EXPERIMENT / 'weights' / 'best.pt'
if not BEST.exists():
    backups = sorted(BACKUP_ROOT.glob('epoch_*/best.pt')) if BACKUP_ROOT.exists() else []
    assert backups, 'best.pt not found in main or backup paths.'
    BEST = backups[-1]

model = YOLO(str(BEST))
onnx_path = model.export(format='onnx', imgsz=IMGSZ, simplify=True)
print('ONNX:', onnx_path)

easy_best = Path('/kaggle/working/best_micro.pt')
shutil.copy2(str(BEST), str(easy_best))
print('Copied easy download file:', easy_best)

print('Download steps:')
print('1. Click Save Version')
print('2. Open Output tab')
print('3. Download /kaggle/working/best_micro.pt or experiments/yolo/.../best.pt')